# Tomography: Figures 6–9 and Table 3

Run the cells in order. Figures, tables, and their data are saved in `results/`.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "src" / "tomography"))
sys.path.insert(0, str(ROOT / "src" / "tomography" / "vendor"))

from compute_backend import DEVICE
print(f"Device: {DEVICE.upper()} (tomography preconditioner; projection and outer solver use CPU)")

from experiment_2d import (save_solver_result, save_raw_results,
                           save_image_data, save_curve_data, save_table_png)
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


import scipy.sparse as sps
from scipy.sparse.linalg import cg as sp_cg

import pandas as pd

from trips.test_problems.Tomography import Tomography
import astra

import jlinops

import numpy as np
import math

from jlinops.base import _CustomLinearOperator
# from .util import get_device

# from ias import IASSolver

from trips.utilities.phantoms import shepp_logan

from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.ticker import FormatStrFormatter

import matplotlib as mpl
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.gridspec as gridspec
# mpl.rcParams['text.usetex'] = True
# mpl.rc('text.latex', preamble=r'\usepackage{amsmath}')

# Define the custom style with RGB color values
custom_style = {
    "axes.prop_cycle": plt.cycler(color=[(106/255, 192/255, 205/255),  # blue - C0
                                         (235/255, 186/255, 63/255),  # gold - C1
                                         (172/255, 46/255, 55/255),  # red - C2
                                         (32/255, 82/255, 145/255),  # dark blue - C3
                                         (83/255, 165/255, 106/255),  # green - C4
                                         (239/255, 145/255, 75/255),  # orange - C5
                                         (208/255, 92/255, 89/255),  # pink - C6
                                         (80/255, 67/255, 157/255),  # purple - C7
                                         (82/255, 82/255, 82/255),  # gray - C8
                                         (53/255, 53/255, 53/255),  # charcoal - C9
                                         (30/255, 27/255, 28/255), # black - C10  
                                         ]) 
}
plt.rc('axes', prop_cycle=custom_style['axes.prop_cycle'])

method_markers = {
    "GKS": "o",
    "S-GKS": "o",
    "resS-GKS": "v",
    "recS-GKS": "^",
    "PS-GKS": "o",
    "resPS-GKS": "v",
    "recPS-GKS": "^",
    "PS-GKB": "o",
    "FGK": "o",
    "FLSQR-I": "$\\text{I}$",
    "FLSQR-R": "$\\text{R}$",
    "FLSQR-W": "$*$",
    "IRW-FLSQR": "x",
}

method_colors = {
    "GKS": "C1",
    "S-GKS": "C4",
    "resS-GKS": "C4",
    "recS-GKS": "C4",
    "PS-GKS": "C2",
    "resPS-GKS": "C2",
    "recPS-GKS": "C2",
    "PS-GKB": "C5",
    "FGK": "C3",
    "FLSQR-I": "C3",
    "FLSQR-R": "C3",
    "FLSQR-W": "C3",
    "IRW-FLSQR": "C3",
}

method_lines = {
    "GKS": "solid",
    "S-GKS": "solid",
    "resS-GKS": "dotted",
    "recS-GKS": "dashed",
    "PS-GKS": "solid",
    "resPS-GKS": "dotted",
    "recPS-GKS": "dashed",
    "PS-GKB": "solid",
    "FGK": "solid",
    "FLSQR-I": "solid",
    "FLSQR-R": "solid",
    "FLSQR-W": "solid",
    "IRW-FLSQR": "solid",
}



import numpy as np

CMAP = "gray"

from msolver import VanillaGKS, MM1GKS, MM2GKS, MM3GKS, IASGKS
from fsolver import MM2FGK, IASFGK
from pgkbsolver import MM3GKB, IASGKB
from pinv import SelectedWeightedNeumann2DPinvOperator


## Setup problem


In [ ]:
np.random.seed(0)
theta_max = 2*np.pi
#Tomo = Tomography (geom="fanflat", CommitCrime=True, theta_max=theta_max)
#Tomo = Tomography (geom="fanflat", CommitCrime=True)
Tomo = Tomography (geom="parallel", CommitCrime=False, theta_max=theta_max)
#views = 28 # until 4/6/25
views = 28 # until 4/6/25
noise_percent = 0.01 # percent of maximum of noiseless signal A x_true
N = 256

x_true = shepp_logan(N)
img_shape = x_true.shape
nx, ny = x_true.shape
(F , b_true , p , q , AforMatrixOperation ) = Tomo.gen_data( x_true , nx , ny , views )

noise_stdev = noise_percent*np.amax(b_true)
noise_var = noise_stdev**2
print(f"Noise level: {noise_percent}")
print(f"Noise stdev: {noise_stdev}")
print(f"Noise var: {noise_var}")


b_noisy = b_true.flatten() + noise_stdev*np.random.normal(size=b_true.flatten().shape)
b_noisy_shaped = b_noisy.reshape((p,q))
noisy_signal = b_noisy.flatten()
m, n = F.shape


# Define linear operator implementing A
_matvec_fn = lambda x: F.matvec(x)
_rmatvec_fn = lambda x: F.rmatvec(x)
A = _CustomLinearOperator( (m,n), _matvec_fn, _rmatvec_fn, device="cpu")

# Underdetermined rate:
underdetermined_rate = noisy_signal.size/(nx*ny)
print(f"Underdetermined rate: {underdetermined_rate}")


In [ ]:
np.savez_compressed(RESULTS / "tomography_data.npz", x_true=x_true,
                    b_true=b_true, b_noisy=b_noisy, b_noisy_shaped=b_noisy_shaped,
                    img_shape=np.asarray(img_shape), noise_var=noise_var,
                    noise_stdev=noise_stdev, views=views,
                    theta_max_requested=theta_max, angles=Tomo.theta,
                    source_origin=Tomo.source_origin, detector_origin=Tomo.detector_origin,
                    detector_spacing=Tomo.detector_pixel_size)


## Figure 6


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.axes_grid1 import make_axes_locatable

# Create a figure and a 2x2 grid of subplots
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(6, 6))

# Display data in each subplot manually with the 'YlGnBu_r' colormap and add a colorbar
im0 = axes[0].imshow(x_true.reshape((nx,ny)), cmap=CMAP)
axes[0].set_title('Ground truth')
#axes[0, 0].axis('off')
axes[0].set_xticks([])
axes[0].set_yticks([])
axes[0].set_xlabel("(a)")
divider = make_axes_locatable(axes[0])
cax0 = divider.append_axes("right", size="5%", pad=0.05)
fig.colorbar(im0, cax=cax0)

im1 = axes[1].imshow(b_noisy_shaped, cmap=CMAP)
axes[1].set_title('Corrupted sinogram $\\mathbf{b}$')
#axes[0, 1].axis('off')
axes[1].set_xticks([])
axes[1].set_yticks([])
axes[1].set_xlabel("(b)")
divider = make_axes_locatable(axes[1])
cax1 = divider.append_axes("right", size="5%", pad=0.05)
fig.colorbar(im1, cax=cax1)

# im2 = axes[1, 0].imshow(np.clip(fbp_recon, a_min=0.0, a_max=None), cmap=CMAP)
# axes[1, 0].set_title('Filtered backprojection')
# # axes[1, 0].axis('off')
# axes[1,0].set_xticks([])
# axes[1,0].set_yticks([])
# axes[1,0].set_xlabel("(c)")
# divider = make_axes_locatable(axes[1, 0])
# cax2 = divider.append_axes("right", size="5%", pad=0.05)
# fig.colorbar(im2, cax=cax2)

# im3 = axes[1, 1].imshow(np.clip(x0.reshape((nx,ny)), a_min=0.0, a_max=None), cmap=CMAP, vmin=0, vmax=1)
# axes[1, 1].set_title('Tikhonov reconstruction')
# #axes[1, 1].axis('off')
# axes[1,1].set_xticks([])
# axes[1,1].set_yticks([])
# axes[1,1].set_xlabel("(d)")
# divider = make_axes_locatable(axes[1, 1])
# cax3 = divider.append_axes("right", size="5%", pad=0.05)
# fig.colorbar(im3, cax=cax3)

# Adjust the layout to make sure there's enough space between subplots
fig.tight_layout()
fig.savefig(RESULTS / "figure_06.png", dpi=250, bbox_inches=None)


# Display the figure
save_image_data(fig, RESULTS / "figure_06.csv")
plt.show()
plt.close(fig)


## Setup prior


In [ ]:
# Set method
Lmat = jlinops.build_neumann2d_sparse_matrix(img_shape)
L = jlinops.Neumann2D(img_shape)

K = np.ones(n)/np.linalg.norm(np.ones(n))
K = jlinops.MatrixLinearOperator(K)


def Lpinv_factory(w, *args, **kwargs):
    return SelectedWeightedNeumann2DPinvOperator(img_shape, weights=w, tol=1e-3, *args, **kwargs)


q = 1.0
mm2_weight_params = {
    "q": q,
}

mm_weight_params = {
    "q": q,
    "epsilon": 1e-4,
}

mmxx_weight_params = {
    "q": q,
    "epsilon": 1e-10,
}

ias_weight_params = {
    "hyperpr_r": 0.5,
    "hyperpr_beta": 3.01,
}


The solver classes use fixed smoothing constants: `MM1GKS` uses 1, `MM2GKS` and `MM2FGK` use $10^{-2}$, and `MM3GKS` and `MM3GKB` use $10^{-3}$. These constants override the `epsilon` entries in the dictionaries above.


## Setup solvers


In [ ]:
# Build solvers
gks_solver = VanillaGKS(A, L, b_noisy, noise_var=noise_var, Lpinv_factory=Lpinv_factory, K=K, img_shape=img_shape)
mm1_gks_solver = MM1GKS(A, L, b_noisy, noise_var=noise_var, Lpinv_factory=Lpinv_factory, K=K, img_shape=img_shape)
mm2_gks_solver = MM2GKS(A, L, b_noisy, noise_var=noise_var, Lpinv_factory=Lpinv_factory, K=K, img_shape=img_shape)
mm3_gks_solver = MM3GKS(A, L, b_noisy, noise_var=noise_var, Lpinv_factory=Lpinv_factory, K=K, img_shape=img_shape)
mm3_gkb_solver = MM3GKB(A, L, b_noisy, noise_var=noise_var, Lpinv_factory=Lpinv_factory, K=K, img_shape=img_shape)
mm2_fgk_solver = MM2FGK(A, L, b_noisy, noise_var=noise_var, Lpinv_factory=Lpinv_factory, K=K, img_shape=img_shape)
ias_gks_solver = IASGKS(A, L, b_noisy, noise_var=noise_var, Lpinv_factory=Lpinv_factory, K=K, img_shape=img_shape)
ias_fgk_solver = IASFGK(A, L, b_noisy, noise_var=noise_var, Lpinv_factory=Lpinv_factory, K=K, img_shape=img_shape)
ias_gkb_solver = IASGKB(A, L, b_noisy, noise_var=noise_var, Lpinv_factory=Lpinv_factory, K=K, img_shape=img_shape)


In [ ]:
# Parameters
method = "dp"
n_its = 100
n_its_recres = 200

restart_dim = 40
rec_max_dim = 40
rec_min_dim = 25

# restart_dim = 35
# rec_max_dim = 35
# rec_min_dim = 20

# restart_dim = 60
# rec_max_dim = 60
# rec_min_dim = 40

ksparsity = None
lambdah_min = 1e-7
lambdah_max = 1e7
fgk_breakdown_tol = 1e-15


## Run all solvers


In [ ]:
all_v_results = []

# GKS
v_gks_data = gks_solver.solve(n_iter=n_its, init_dim=5, x0=None, lambdah0=1e0, lambdah_min=lambdah_min, lambdah_max=lambdah_max,
                                    regparam=method, x_true=x_true.flatten(), weight_params={},
                                    regparam_kwargs={"eta":1.01})

all_v_results.append( ( "V", "GKS", v_gks_data  ) )

save_solver_result(RESULTS / "tomography" / "v_gks.npz", v_gks_data)


### MM weights


In [ ]:
all_mm_results = []


In [ ]:
# S-GKS (2)
mm_sgks_data = mm2_gks_solver.solve(n_iter=n_its, init_dim=5, x0=v_gks_data["x"], lambdah0=v_gks_data["lambdah"], lambdah_min=lambdah_min, lambdah_max=lambdah_max,
                                    regparam=method, x_true=x_true.flatten(), weight_params=mm2_weight_params,
                                    regparam_kwargs={"eta":1.01})

all_mm_results.append( ( "MM", "S-GKS", mm_sgks_data  ) )

save_solver_result(RESULTS / "tomography" / "mm_sgks.npz", mm_sgks_data)


In [ ]:
# Recycled S-GKS (2)
mm_recsgks_data = mm2_gks_solver.solve(n_iter=n_its_recres, init_dim=5, x0=v_gks_data["x"], lambdah0=v_gks_data["lambdah"], lambdah_min=lambdah_min, lambdah_max=lambdah_max,
                                    regparam=method, x_true=x_true.flatten(), weight_params=mm2_weight_params,
                                    regparam_kwargs={"eta":1.01}, restart_dim=None, min_dim=rec_min_dim, max_dim=rec_max_dim)

all_mm_results.append( ( "MM", "S-GKS (recycled)", mm_recsgks_data  ) )

save_solver_result(RESULTS / "tomography" / "mm_recsgks.npz", mm_recsgks_data)


In [ ]:
# Restarted S-GKS (1)
mm_ressgks_data = mm1_gks_solver.solve(n_iter=n_its_recres, init_dim=5, x0=v_gks_data["x"], lambdah0=v_gks_data["lambdah"], lambdah_min=lambdah_min, lambdah_max=lambdah_max,
                                    regparam=method, x_true=x_true.flatten(), weight_params=mm2_weight_params,
                                    regparam_kwargs={"eta":1.01}, restart_dim=restart_dim)

all_mm_results.append( ( "MM", "S-GKS (restarted)", mm_ressgks_data  ) )

save_solver_result(RESULTS / "tomography" / "mm_ressgks.npz", mm_ressgks_data)


In [ ]:
# PS-GKS
mm_psgks_data = mm3_gks_solver.solve_with_priorconditioning_mod(n_iter=n_its, init_dim=5, x0=v_gks_data["x"], lambdah0=v_gks_data["lambdah"], lambdah_min=lambdah_min, lambdah_max=lambdah_max,
                                                 regparam=method, x_true=x_true.flatten(), weight_params=mm_weight_params,
                                                 regparam_kwargs={"eta":1.01})

all_mm_results.append( ( "MM", "PS-GKS", mm_psgks_data  ) )

save_solver_result(RESULTS / "tomography" / "mm_psgks.npz", mm_psgks_data)


In [ ]:
# Restarted PS-GKS
mm_respsgks_data = mm3_gks_solver.solve_with_priorconditioning_mod(n_iter=n_its_recres, init_dim=5, x0=v_gks_data["x"], lambdah0=v_gks_data["lambdah"], lambdah_min=lambdah_min, lambdah_max=lambdah_max,
                                                 regparam=method, x_true=x_true.flatten(), weight_params=mm_weight_params, restart_dim=restart_dim,
                                                 regparam_kwargs={"eta":1.01})

all_mm_results.append( ( "MM", "PS-GKS (restarted)", mm_respsgks_data  ) )

save_solver_result(RESULTS / "tomography" / "mm_respsgks.npz", mm_respsgks_data)


In [ ]:
# Recycled PS-GKS
mm_recpsgks_data = mm3_gks_solver.solve_with_priorconditioning_mod(n_iter=n_its_recres, init_dim=5, x0=v_gks_data["x"], lambdah0=v_gks_data["lambdah"], lambdah_min=lambdah_min, lambdah_max=lambdah_max,
                                                 regparam=method, x_true=x_true.flatten(), weight_params=mm_weight_params, min_dim=rec_min_dim, max_dim=rec_max_dim,
                                                 regparam_kwargs={"eta":1.01})

all_mm_results.append( ( "MM", "PS-GKS (recycled)", mm_recpsgks_data  ) )

save_solver_result(RESULTS / "tomography" / "mm_recpsgks.npz", mm_recpsgks_data)


In [ ]:
# PS-GKB solver
mm_gkb_data = mm3_gkb_solver.solve(n_iter=n_its, x0=None, lambdah0=None,
                                                 regparam=method, x_true=x_true.flatten(), ksparsity=None, weight_params=mm_weight_params,
                                                 lambdah_min=lambdah_min, lambdah_max=lambdah_max,
                                                #  regparam_kwargs={"eta":1.01},
                                                #  lambdah_min=1e-7, lambdah_max=1e7,
                                                #  init_dim=5
                                                 )
all_mm_results.append( ( "MM", "PS-GKB", mm_gkb_data  ) )

save_solver_result(RESULTS / "tomography" / "mm_gkb.npz", mm_gkb_data)


In [ ]:
# FGK solver (with 2, trying it out)
mm_fgk_data = mm2_fgk_solver.solve(n_iter=n_its, x0=None, lambdah0=None, breakdown_tol=fgk_breakdown_tol,
                                                 regparam=method, x_true=x_true.flatten(), ksparsity=None, weight_params=mmxx_weight_params, refresh_basis=False, lambdah_min=lambdah_min, lambdah_max=lambdah_max,
                                                #  regparam_kwargs={"eta":1.01},
                                                #  lambdah_min=1e-7, lambdah_max=1e7,
                                                #  init_dim=5
                                                 )

save_solver_result(RESULTS / "tomography" / "mm_fgk.npz", mm_fgk_data)


### IAS weights


In [ ]:
all_ias_results = []


In [ ]:
# S-GKS
ias_sgks_data = ias_gks_solver.solve(n_iter=n_its, init_dim=5, x0=v_gks_data["x"], lambdah0=v_gks_data["lambdah"], lambdah_min=lambdah_min, lambdah_max=lambdah_max,
                                    regparam=method, x_true=x_true.flatten(), weight_params=ias_weight_params,
                                    regparam_kwargs={"eta":1.01})

all_ias_results.append( ( "IAS", "S-GKS", ias_sgks_data  ) )

save_solver_result(RESULTS / "tomography" / "ias_sgks.npz", ias_sgks_data)


In [ ]:
# Restarted S-GKS
ias_ressgks_data = ias_gks_solver.solve(n_iter=n_its_recres, init_dim=5, x0=v_gks_data["x"], lambdah0=v_gks_data["lambdah"], lambdah_min=lambdah_min, lambdah_max=lambdah_max,
                                    regparam=method, x_true=x_true.flatten(), weight_params=ias_weight_params,
                                    regparam_kwargs={"eta":1.01}, restart_dim=None, min_dim=rec_min_dim, max_dim=rec_max_dim)

all_ias_results.append( ( "IAS", "S-GKS (restarted)", ias_ressgks_data  ) )

save_solver_result(RESULTS / "tomography" / "ias_ressgks.npz", ias_ressgks_data)


In [ ]:
# Recycled S-GKS
ias_recsgks_data = ias_gks_solver.solve(n_iter=n_its_recres, init_dim=5, x0=v_gks_data["x"], lambdah0=v_gks_data["lambdah"], lambdah_min=lambdah_min, lambdah_max=lambdah_max,
                                    regparam=method, x_true=x_true.flatten(), weight_params=ias_weight_params,
                                    regparam_kwargs={"eta":1.01}, restart_dim=restart_dim)

all_ias_results.append( ( "IAS", "S-GKS (recycled)", ias_recsgks_data  ) )

save_solver_result(RESULTS / "tomography" / "ias_recsgks.npz", ias_recsgks_data)


In [ ]:
# PS-GKS
ias_psgks_data = ias_gks_solver.solve_with_priorconditioning_mod(n_iter=n_its, init_dim=5, x0=v_gks_data["x"], lambdah0=v_gks_data["lambdah"], lambdah_min=lambdah_min, lambdah_max=lambdah_max,
                                                 regparam=method, x_true=x_true.flatten(), weight_params=ias_weight_params,
                                                 regparam_kwargs={"eta":1.01})

all_ias_results.append( ( "IAS", "PS-GKS", ias_psgks_data  ) )

save_solver_result(RESULTS / "tomography" / "ias_psgks.npz", ias_psgks_data)


In [ ]:
# Restarted PS-GKS
ias_respsgks_data = ias_gks_solver.solve_with_priorconditioning_mod(n_iter=n_its_recres, init_dim=5, x0=v_gks_data["x"], lambdah0=v_gks_data["lambdah"], lambdah_min=lambdah_min, lambdah_max=lambdah_max,
                                                 regparam=method, x_true=x_true.flatten(), weight_params=ias_weight_params, restart_dim=restart_dim,
                                                 regparam_kwargs={"eta":1.01})

all_ias_results.append( ( "IAS", "PS-GKS (restarted)", ias_psgks_data  ) )

save_solver_result(RESULTS / "tomography" / "ias_respsgks.npz", ias_respsgks_data)


In [ ]:
# Recycled PS-GKS
ias_recpsgks_data = ias_gks_solver.solve_with_priorconditioning_mod(n_iter=n_its_recres, init_dim=5, x0=v_gks_data["x"], lambdah0=v_gks_data["lambdah"], lambdah_min=lambdah_min, lambdah_max=lambdah_max,
                                                 regparam=method, x_true=x_true.flatten(), weight_params=ias_weight_params, min_dim=rec_min_dim, max_dim=rec_max_dim,
                                                 regparam_kwargs={"eta":1.01})

all_ias_results.append( ( "IAS", "PS-GKS (recycled)", ias_recpsgks_data  ) )

save_solver_result(RESULTS / "tomography" / "ias_recpsgks.npz", ias_recpsgks_data)


In [ ]:
# PS-GKB solver
ias_gkb_data = ias_gkb_solver.solve(n_iter=n_its, x0=None, lambdah0=None,
                                                 regparam=method, x_true=x_true.flatten(), ksparsity=None, weight_params=ias_weight_params,
                                                 lambdah_min=lambdah_min, lambdah_max=lambdah_max,
                                                #  regparam_kwargs={"eta":1.01},
                                                #  lambdah_min=1e-7, lambdah_max=1e7,
                                                #  init_dim=5
                                                 )
all_ias_results.append( ( "IAS", "PS-GKB", ias_gkb_data  ) )

save_solver_result(RESULTS / "tomography" / "ias_gkb.npz", ias_gkb_data)


In [ ]:
# FGK solver
ias_fgk_data = ias_fgk_solver.solve(n_iter=n_its, x0=None, lambdah0=None, breakdown_tol=fgk_breakdown_tol,
                                                 regparam=method, x_true=x_true.flatten(), ksparsity=None, weight_params=ias_weight_params, refresh_basis=False, lambdah_min=lambdah_min, lambdah_max=lambdah_max,
                                                #  regparam_kwargs={"eta":1.01},
                                                #  lambdah_min=1e-7, lambdah_max=1e7,
                                                #  init_dim=5
                                                 )

all_ias_results.append( ( "IAS", "FGK", ias_fgk_data  ) )

save_solver_result(RESULTS / "tomography" / "ias_fgk.npz", ias_fgk_data)


## Table 3


In [ ]:
# Refresh if needed
all_mm_results = [
    ( "MM", "S-GKS",  mm_sgks_data),
    ("MM", "resS-GKS", mm_ressgks_data),
    ("MM", "recS-GKS", mm_recsgks_data),
    ("MM", "PS-GKS", mm_psgks_data),
    ("MM", "resPS-GKS", mm_respsgks_data),
    ("MM", "recPS-GKS", mm_recpsgks_data),
    ("MM", "PS-GKB", mm_gkb_data),
    ("MM", "FGK", mm_fgk_data),
]

all_ias_results = [
    ( "IAS", "S-GKS",  ias_sgks_data),
    ("IAS", "resS-GKS", ias_ressgks_data),
    ("IAS", "recS-GKS", ias_recsgks_data),
    ("IAS", "PS-GKS", ias_psgks_data),
    ("IAS", "resPS-GKS", ias_respsgks_data),
    ("IAS", "recPS-GKS", ias_recpsgks_data),
    ("IAS", "PS-GKB", ias_gkb_data),
    ("IAS", "FGK", ias_fgk_data),
]


In [ ]:
v_df = pd.DataFrame({"Weights":[], "Method":[], "lambdah":[], "RRE":[], "SSIM": [], "HaarPSI":[], "Gini index": [], "Hoyer index": [], "A matvecs": [], "L matvecs": [], "WLpinv matvecs": [], "n_iters": []   })

for set in all_v_results:
        weight_str, method_label, method_data = set
        new_df = pd.DataFrame({"Weights":[weight_str], "Method":[method_label], "lambdah":[ method_data["lambda_history"][-1] ], "RRE":[ method_data["rre_history"][-1] ], "SSIM": [ method_data["ssim_history"][-1] ], "HaarPSI":[ method_data["haar_history"][-1] ], "Gini index": [ method_data["gini_indices"][-1] ], "Hoyer index": [ method_data["hoyer_indices"][-1] ] , "A matvecs": [ method_data["tot_matvec_A"] ], "L matvecs": [ method_data["tot_matvec_L"] ], "WLpinv matvecs": [ method_data["tot_matvec_WLpinv"] ], "n_iters": [ method_data["n_completed_iterations"] ]   })
        v_df = pd.concat([v_df, new_df], axis=0)


mm_df = pd.DataFrame({"Weights":[], "Method":[], "lambdah":[], "RRE":[], "SSIM": [], "Gini index": [], "Hoyer index": []})

for set in all_mm_results:
        weight_str, method_label, method_data = set
        # print(method_label)
        # print(method_data["rre_history"][-1])
        new_df = pd.DataFrame({"Weights":[weight_str], "Method":[method_label], "lambdah":[ method_data["lambda_history"][-1] ], "RRE":[ method_data["rre_history"][-1] ], "SSIM": [ method_data["ssim_history"][-1] ], "HaarPSI":[ method_data["haar_history"][-1] ], "Gini index": [ method_data["gini_indices"][-1] ], "Hoyer index": [ method_data["hoyer_indices"][-1] ],  "A matvecs": [ method_data["tot_matvec_A"] ], "L matvecs": [ method_data["tot_matvec_L"] ], "WLpinv matvecs": [ method_data["tot_matvec_WLpinv"] ], "n_iters": [ method_data["n_completed_iterations"] ]   })
        mm_df = pd.concat([mm_df, new_df], axis=0)



ias_df = pd.DataFrame({"Weights":[], "Method":[], "lambdah":[], "RRE":[], "SSIM": [], "Gini index": [], "Hoyer index": []})

for set in all_ias_results:
        weight_str, method_label, method_data = set
        new_df = pd.DataFrame({"Weights":[weight_str], "Method":[method_label], "lambdah":[ method_data["lambda_history"][-1] ], "RRE":[ method_data["rre_history"][-1] ], "SSIM": [ method_data["ssim_history"][-1] ], "HaarPSI":[ method_data["haar_history"][-1] ], "Gini index": [ method_data["gini_indices"][-1] ], "Hoyer index": [ method_data["hoyer_indices"][-1] ],  "A matvecs": [ method_data["tot_matvec_A"] ], "L matvecs": [ method_data["tot_matvec_L"] ], "WLpinv matvecs": [ method_data["tot_matvec_WLpinv"] ], "n_iters": [ method_data["n_completed_iterations"] ]    })
        ias_df = pd.concat([ias_df, new_df], axis=0)


In [ ]:
# all_df = pd.concat([v_df, mm_df, ias_df])
# all_df = all_df.rename(columns={'lambdah': '$\lambda$'})
# all_df

all_df = pd.concat([v_df, mm_df, ias_df])
all_df = all_df[["Weights", "Method", "lambdah", "n_iters", "RRE", "SSIM", "Gini index", "A matvecs", "L matvecs", "WLpinv matvecs"]]
all_df = all_df.rename(columns={'lambdah': '$\lambda$', "n_iters":"$n_{\\textrm{iter}}$", "A matvecs":"$n_{\mathbf{A}}$", "L matvecs":"$n_{\mathbf{L}}$", "WLpinv matvecs":"$n_{\mathbf{ L }_k^\dagger }$"  })

all_df.to_csv(RESULTS / "table_03.csv", index=False)
save_table_png(all_df, RESULTS / "table_03.png")
all_df


In [ ]:
def scientific_to_latex(sci_str):
    # Split the number into base and exponent parts
    sci_str = f"{sci_str:.0e}"
    base, exponent = sci_str.split('e')
    
    # Convert base to integer (since the decimal places can be ignored)
    base = str(int(float(base)))
    
    # Prepare the exponent part, removing the leading "+" if it's a positive exponent
    exponent = exponent.lstrip("+")
    
    # Format the result as a LaTeX string
    latex_str = f"${base}\\cdot10^{{{int(exponent)}}}$"
    
    return latex_str


In [ ]:
print( all_df.to_latex(index=False, formatters={'$\lambda$': lambda s: scientific_to_latex(s), 
                                                "RRE": "${:.3f}$", "SSIM": "${:.3f}$", 
                                                  "Gini index": "${:.3f}$", "Hoyer index": "${:.3f}$", 
                                                  "$n_{\\textrm{iter}}$": "${:.0f}$",
                                                  "$n_{\mathbf{A}}$": "${:.0f}$", "$n_{\mathbf{L}}$": "${:.0f}$", "$n_{\mathbf{ L }_k^\dagger }$": "${:.0f}$"
                                                    }) )


In [ ]:
save_raw_results(RESULTS / "tomography_raw.npz", all_v_results + all_mm_results + all_ias_results)


## Figure 7


In [ ]:
err_cmap = "binary"

vbound = 0.4
metric = "ssim_history"
ground_truth = x_true.copy()
b = noisy_signal.copy()

fig, axs = plt.subplots(4,4,figsize=(10,10))

# Observed
# perf_metric = structural_similarity( b.reshape(img_shape), ground_truth.reshape(img_shape), data_range=(np.amax(ground_truth) - np.amin(ground_truth) ), gaussian_weights=True, sigma=1.5, use_sample_covariance=False )
# axs[0,0].imshow(b.reshape(img_shape), cmap="gray", vmin=0, vmax=1)
# axs[0,0].set_title(f"Data ({perf_metric:.2f})")
# axs[0,0].set_xticks([])
# axs[0,0].set_yticks([])

# axs[1,0].imshow(np.abs( b.reshape(img_shape) - ground_truth.reshape(img_shape) ), cmap=err_cmap, vmin=0, vmax=vbound)
# axs[1,0].set_xticks([])
# axs[1,0].set_yticks([])


# V-GKS
axs[0,0].imshow(v_gks_data["x"].reshape(img_shape), cmap="gray", vmin=0, vmax=1)
perf_metric = v_gks_data[metric][-1]
axs[0,0].set_title(f"GKS ({perf_metric:.2f})")
axs[0,0].set_xticks([])
axs[0,0].set_yticks([])

axs[1,0].imshow(np.abs( v_gks_data["x"].reshape(img_shape) - ground_truth.reshape(img_shape) ), cmap=err_cmap, vmin=0, vmax=vbound)
axs[1,0].set_xticks([])
axs[1,0].set_yticks([])

# FGK
axs[2,0].imshow(mm_fgk_data["x"].reshape(img_shape), cmap="gray", vmin=0, vmax=1)
perf_metric = mm_fgk_data[metric][-1]
axs[2,0].set_title(f"FGK ({perf_metric:.2f})")
axs[2,0].set_xticks([])
axs[2,0].set_yticks([])

axs[3,0].imshow(np.abs( mm_fgk_data["x"].reshape(img_shape) - ground_truth.reshape(img_shape) ), cmap=err_cmap, vmin=0, vmax=vbound)
axs[3,0].set_xticks([])
axs[3,0].set_yticks([])




# S-GKS
axs[0,1].imshow(mm_sgks_data["x"].reshape(img_shape), cmap="gray", vmin=0, vmax=1)
perf_metric = mm_sgks_data[metric][-1]
axs[0,1].set_title(f"S-GKS ({perf_metric:.2f})")
axs[0,1].set_xticks([])
axs[0,1].set_yticks([])

axs[1,1].imshow(np.abs( mm_sgks_data["x"].reshape(img_shape) - ground_truth.reshape(img_shape) ), cmap=err_cmap, vmin=0, vmax=vbound)
axs[1,1].set_xticks([])
axs[1,1].set_yticks([])



# PS-GKS
axs[2,1].imshow(mm_psgks_data["x"].reshape(img_shape), cmap="gray", vmin=0, vmax=1)
perf_metric = mm_psgks_data[metric][-1]
axs[2,1].set_title(f"PS-GKS ({perf_metric:.2f})")
axs[2,1].set_xticks([])
axs[2,1].set_yticks([])

axs[3,1].imshow(np.abs( mm_psgks_data["x"].reshape(img_shape) - ground_truth.reshape(img_shape) ), cmap=err_cmap, vmin=0, vmax=vbound)
axs[3,1].set_xticks([])
axs[3,1].set_yticks([])




# S-GKS (restarted)
axs[0,2].imshow(mm_ressgks_data["x"].reshape(img_shape), cmap="gray", vmin=0, vmax=1)
perf_metric = mm_ressgks_data[metric][-1]
axs[0,2].set_title(f"resS-GKS ({perf_metric:.2f})")
axs[0,2].set_xticks([])
axs[0,2].set_yticks([])

axs[1,2].imshow(np.abs( mm_ressgks_data["x"].reshape(img_shape) - ground_truth.reshape(img_shape) ), cmap=err_cmap, vmin=0, vmax=vbound)
axs[1,2].set_xticks([])
axs[1,2].set_yticks([])



# PS-GKS (restarted)
axs[2,2].imshow(mm_respsgks_data["x"].reshape(img_shape), cmap="gray", vmin=0, vmax=1)
perf_metric = mm_respsgks_data[metric][-1]
axs[2,2].set_title(f"resPS-GKS ({perf_metric:.2f})")
axs[2,2].set_xticks([])
axs[2,2].set_yticks([])

axs[3,2].imshow(np.abs( mm_respsgks_data["x"].reshape(img_shape) - ground_truth.reshape(img_shape) ), cmap=err_cmap, vmin=0, vmax=vbound)
axs[3,2].set_xticks([])
axs[3,2].set_yticks([])






# S-GKS (recycling)
axs[0,3].imshow(mm_recsgks_data["x"].reshape(img_shape), cmap="gray", vmin=0, vmax=1)
perf_metric = mm_recsgks_data[metric][-1]
axs[0,3].set_title(f"recS-GKS ({perf_metric:.2f})")
axs[0,3].set_xticks([])
axs[0,3].set_yticks([])

axs[1,3].imshow(np.abs( mm_recsgks_data["x"].reshape(img_shape) - ground_truth.reshape(img_shape) ), cmap=err_cmap, vmin=0, vmax=vbound)
axs[1,3].set_xticks([])
axs[1,3].set_yticks([])



# PS-GKS (recycling)
axs[2,3].imshow(mm_recpsgks_data["x"].reshape(img_shape), cmap="gray", vmin=0, vmax=1)
perf_metric = mm_recpsgks_data[metric][-1]
axs[2,3].set_title(f"recPS-GKS ({perf_metric:.2f})")
axs[2,3].set_xticks([])
axs[2,3].set_yticks([])

axs[3,3].imshow(np.abs( mm_recpsgks_data["x"].reshape(img_shape) - ground_truth.reshape(img_shape) ), cmap=err_cmap, vmin=0, vmax=vbound)
axs[3,3].set_xticks([])
axs[3,3].set_yticks([])









# # PS-GKB
# axs[0,3].imshow(mm_psgkb_data["x"].reshape(img_shape), cmap="gray", vmin=0, vmax=1)
# perf_metric = mm_psgkb_data[metric][-1]
# axs[0,3].set_title(f"PS-GKB ({perf_metric:.2f})")
# axs[0,3].set_xticks([])
# axs[0,3].set_yticks([])

# axs[1,3].imshow(np.abs( mm_psgkb_data["x"].reshape(img_shape) - ground_truth.reshape(img_shape) ), cmap=err_cmap, vmin=0, vmax=vbound)
# axs[1,3].set_xticks([])
# axs[1,3].set_yticks([])



# # LSQR-I
# axs[2,1].imshow(mm_fgk2019I_data["x"].reshape(img_shape), cmap="gray", vmin=0, vmax=1)
# perf_metric = mm_fgk2019I_data[metric][-1]
# axs[2,1].set_title(f"FLSQR-I ({perf_metric:.2f})")
# axs[2,1].set_xticks([])
# axs[2,1].set_yticks([])

# axs[3,1].imshow(np.abs( mm_fgk2019I_data["x"].reshape(img_shape) - ground_truth.reshape(img_shape) ), cmap=err_cmap, vmin=0, vmax=vbound)
# axs[3,1].set_xticks([])
# axs[3,1].set_yticks([])



# # LSQR-R
# axs[2,2].imshow(mm_fgk2019R_data["x"].reshape(img_shape), cmap="gray", vmin=0, vmax=1)
# perf_metric = mm_fgk2019R_data[metric][-1]
# axs[2,2].set_title(f"FLSQR-R ({perf_metric:.2f})")
# axs[2,2].set_xticks([])
# axs[2,2].set_yticks([])

# axs[3,2].imshow(np.abs( mm_fgk2019R_data["x"].reshape(img_shape) - ground_truth.reshape(img_shape) ), cmap=err_cmap, vmin=0, vmax=vbound)
# axs[3,2].set_xticks([])
# axs[3,2].set_yticks([])


# # LSQR-W
# axs[2,3].imshow(mm_fgk2019W_data["x"].reshape(img_shape), cmap="gray", vmin=0, vmax=1)
# perf_metric = mm_fgk2019W_data[metric][-1]
# axs[2,3].set_title(f"FLSQR-W ({perf_metric:.2f})")
# axs[2,3].set_xticks([])
# axs[2,3].set_yticks([])

# axs[3,3].imshow(np.abs( mm_fgk2019W_data["x"].reshape(img_shape) - ground_truth.reshape(img_shape) ), cmap=err_cmap, vmin=0, vmax=vbound)
# axs[3,3].set_xticks([])
# axs[3,3].set_yticks([])




# # IRW-LSQR
# axs[2,4].imshow(mm_fgk2019IRW_data["x"].reshape(img_shape), cmap="gray", vmin=0, vmax=1)
# perf_metric = mm_fgk2019IRW_data[metric][-1]
# axs[2,4].set_title(f"IRW-FLSQR ({perf_metric:.2f})")
# axs[2,4].set_xticks([])
# axs[2,4].set_yticks([])

# axs[3,4].imshow(np.abs( mm_fgk2019IRW_data["x"].reshape(img_shape) - ground_truth.reshape(img_shape) ), cmap=err_cmap, vmin=0, vmax=vbound)
# axs[3,4].set_xticks([])
# axs[3,4].set_yticks([])


fig.tight_layout()

fig.savefig(RESULTS / "figure_07.png", dpi=250, bbox_inches=None)


save_image_data(fig, RESULTS / "figure_07.csv")
plt.show()
plt.close(fig)


## Figure 8


In [ ]:
err_cmap = "binary"

vbound = 0.4
metric = "ssim_history"
ground_truth = x_true.copy()
b = noisy_signal.copy()

fig, axs = plt.subplots(4,4,figsize=(10,10))

# Observed
# perf_metric = structural_similarity( b.reshape(img_shape), ground_truth.reshape(img_shape), data_range=(np.amax(ground_truth) - np.amin(ground_truth) ), gaussian_weights=True, sigma=1.5, use_sample_covariance=False )
# axs[0,0].imshow(b.reshape(img_shape), cmap="gray", vmin=0, vmax=1)
# axs[0,0].set_title(f"Data ({perf_metric:.2f})")
# axs[0,0].set_xticks([])
# axs[0,0].set_yticks([])

# axs[1,0].imshow(np.abs( b.reshape(img_shape) - ground_truth.reshape(img_shape) ), cmap=err_cmap, vmin=0, vmax=vbound)
# axs[1,0].set_xticks([])
# axs[1,0].set_yticks([])


# V-GKS
axs[0,0].imshow(v_gks_data["x"].reshape(img_shape), cmap="gray", vmin=0, vmax=1)
perf_metric = v_gks_data[metric][-1]
axs[0,0].set_title(f"GKS ({perf_metric:.2f})")
axs[0,0].set_xticks([])
axs[0,0].set_yticks([])

axs[1,0].imshow(np.abs( v_gks_data["x"].reshape(img_shape) - ground_truth.reshape(img_shape) ), cmap=err_cmap, vmin=0, vmax=vbound)
axs[1,0].set_xticks([])
axs[1,0].set_yticks([])

# FGK
axs[2,0].imshow(ias_fgk_data["x"].reshape(img_shape), cmap="gray", vmin=0, vmax=1)
perf_metric = ias_fgk_data[metric][-1]
axs[2,0].set_title(f"FGK ({perf_metric:.2f})")
axs[2,0].set_xticks([])
axs[2,0].set_yticks([])

axs[3,0].imshow(np.abs( ias_fgk_data["x"].reshape(img_shape) - ground_truth.reshape(img_shape) ), cmap=err_cmap, vmin=0, vmax=vbound)
axs[3,0].set_xticks([])
axs[3,0].set_yticks([])




# S-GKS
axs[0,1].imshow(ias_sgks_data["x"].reshape(img_shape), cmap="gray", vmin=0, vmax=1)
perf_metric = ias_sgks_data[metric][-1]
axs[0,1].set_title(f"S-GKS ({perf_metric:.2f})")
axs[0,1].set_xticks([])
axs[0,1].set_yticks([])

axs[1,1].imshow(np.abs( ias_sgks_data["x"].reshape(img_shape) - ground_truth.reshape(img_shape) ), cmap=err_cmap, vmin=0, vmax=vbound)
axs[1,1].set_xticks([])
axs[1,1].set_yticks([])



# PS-GKS
axs[2,1].imshow(ias_psgks_data["x"].reshape(img_shape), cmap="gray", vmin=0, vmax=1)
perf_metric = ias_psgks_data[metric][-1]
axs[2,1].set_title(f"PS-GKS ({perf_metric:.2f})")
axs[2,1].set_xticks([])
axs[2,1].set_yticks([])

axs[3,1].imshow(np.abs( ias_psgks_data["x"].reshape(img_shape) - ground_truth.reshape(img_shape) ), cmap=err_cmap, vmin=0, vmax=vbound)
axs[3,1].set_xticks([])
axs[3,1].set_yticks([])




# S-GKS (restarted)
axs[0,2].imshow(ias_ressgks_data["x"].reshape(img_shape), cmap="gray", vmin=0, vmax=1)
perf_metric = ias_ressgks_data[metric][-1]
axs[0,2].set_title(f"resS-GKS ({perf_metric:.2f})")
axs[0,2].set_xticks([])
axs[0,2].set_yticks([])

axs[1,2].imshow(np.abs( ias_ressgks_data["x"].reshape(img_shape) - ground_truth.reshape(img_shape) ), cmap=err_cmap, vmin=0, vmax=vbound)
axs[1,2].set_xticks([])
axs[1,2].set_yticks([])



# PS-GKS (restarted)
axs[2,2].imshow(ias_respsgks_data["x"].reshape(img_shape), cmap="gray", vmin=0, vmax=1)
perf_metric = ias_respsgks_data[metric][-1]
axs[2,2].set_title(f"resPS-GKS ({perf_metric:.2f})")
axs[2,2].set_xticks([])
axs[2,2].set_yticks([])

axs[3,2].imshow(np.abs( ias_respsgks_data["x"].reshape(img_shape) - ground_truth.reshape(img_shape) ), cmap=err_cmap, vmin=0, vmax=vbound)
axs[3,2].set_xticks([])
axs[3,2].set_yticks([])






# S-GKS (recycling)
axs[0,3].imshow(ias_recsgks_data["x"].reshape(img_shape), cmap="gray", vmin=0, vmax=1)
perf_metric = ias_recsgks_data[metric][-1]
axs[0,3].set_title(f"recS-GKS ({perf_metric:.2f})")
axs[0,3].set_xticks([])
axs[0,3].set_yticks([])

axs[1,3].imshow(np.abs( ias_recsgks_data["x"].reshape(img_shape) - ground_truth.reshape(img_shape) ), cmap=err_cmap, vmin=0, vmax=vbound)
axs[1,3].set_xticks([])
axs[1,3].set_yticks([])



# PS-GKS (recycling)
axs[2,3].imshow(ias_recpsgks_data["x"].reshape(img_shape), cmap="gray", vmin=0, vmax=1)
perf_metric = ias_recpsgks_data[metric][-1]
axs[2,3].set_title(f"recPS-GKS ({perf_metric:.2f})")
axs[2,3].set_xticks([])
axs[2,3].set_yticks([])

axs[3,3].imshow(np.abs( ias_recpsgks_data["x"].reshape(img_shape) - ground_truth.reshape(img_shape) ), cmap=err_cmap, vmin=0, vmax=vbound)
axs[3,3].set_xticks([])
axs[3,3].set_yticks([])









# # PS-GKB
# axs[0,3].imshow(mm_psgkb_data["x"].reshape(img_shape), cmap="gray", vmin=0, vmax=1)
# perf_metric = mm_psgkb_data[metric][-1]
# axs[0,3].set_title(f"PS-GKB ({perf_metric:.2f})")
# axs[0,3].set_xticks([])
# axs[0,3].set_yticks([])

# axs[1,3].imshow(np.abs( mm_psgkb_data["x"].reshape(img_shape) - ground_truth.reshape(img_shape) ), cmap=err_cmap, vmin=0, vmax=vbound)
# axs[1,3].set_xticks([])
# axs[1,3].set_yticks([])



# # LSQR-I
# axs[2,1].imshow(mm_fgk2019I_data["x"].reshape(img_shape), cmap="gray", vmin=0, vmax=1)
# perf_metric = mm_fgk2019I_data[metric][-1]
# axs[2,1].set_title(f"FLSQR-I ({perf_metric:.2f})")
# axs[2,1].set_xticks([])
# axs[2,1].set_yticks([])

# axs[3,1].imshow(np.abs( mm_fgk2019I_data["x"].reshape(img_shape) - ground_truth.reshape(img_shape) ), cmap=err_cmap, vmin=0, vmax=vbound)
# axs[3,1].set_xticks([])
# axs[3,1].set_yticks([])



# # LSQR-R
# axs[2,2].imshow(mm_fgk2019R_data["x"].reshape(img_shape), cmap="gray", vmin=0, vmax=1)
# perf_metric = mm_fgk2019R_data[metric][-1]
# axs[2,2].set_title(f"FLSQR-R ({perf_metric:.2f})")
# axs[2,2].set_xticks([])
# axs[2,2].set_yticks([])

# axs[3,2].imshow(np.abs( mm_fgk2019R_data["x"].reshape(img_shape) - ground_truth.reshape(img_shape) ), cmap=err_cmap, vmin=0, vmax=vbound)
# axs[3,2].set_xticks([])
# axs[3,2].set_yticks([])


# # LSQR-W
# axs[2,3].imshow(mm_fgk2019W_data["x"].reshape(img_shape), cmap="gray", vmin=0, vmax=1)
# perf_metric = mm_fgk2019W_data[metric][-1]
# axs[2,3].set_title(f"FLSQR-W ({perf_metric:.2f})")
# axs[2,3].set_xticks([])
# axs[2,3].set_yticks([])

# axs[3,3].imshow(np.abs( mm_fgk2019W_data["x"].reshape(img_shape) - ground_truth.reshape(img_shape) ), cmap=err_cmap, vmin=0, vmax=vbound)
# axs[3,3].set_xticks([])
# axs[3,3].set_yticks([])




# # IRW-LSQR
# axs[2,4].imshow(mm_fgk2019IRW_data["x"].reshape(img_shape), cmap="gray", vmin=0, vmax=1)
# perf_metric = mm_fgk2019IRW_data[metric][-1]
# axs[2,4].set_title(f"IRW-FLSQR ({perf_metric:.2f})")
# axs[2,4].set_xticks([])
# axs[2,4].set_yticks([])

# axs[3,4].imshow(np.abs( mm_fgk2019IRW_data["x"].reshape(img_shape) - ground_truth.reshape(img_shape) ), cmap=err_cmap, vmin=0, vmax=vbound)
# axs[3,4].set_xticks([])
# axs[3,4].set_yticks([])


fig.tight_layout()

fig.savefig(RESULTS / "figure_08.png", dpi=250, bbox_inches=None)


save_image_data(fig, RESULTS / "figure_08.csv")
plt.show()
plt.close(fig)


## Figure 9


In [ ]:
metric = "gini_indices"
#metric = "hoyer_indices"
# metric = "rre_history"
start_idx = 5
marker_size = 10.0
line_width = 2.0


method_lines = {
    "GKS": "solid",
    "S-GKS": "solid",
    "resS-GKS": "dotted",
    "recS-GKS": "dashed",
    "PS-GKS": "solid",
    "resPS-GKS": "dotted",
    "recPS-GKS": "dashed",
    "PS-GKB": "dashed",
    "FGK": "dotted",
    "FLSQR-I": "dotted",
    "FLSQR-R": "dashed",
    "FLSQR-W": "dashdot",
    "IRW-FLSQR": (0, (3, 1, 1, 1, 1, 1)),
}

fig, axs = plt.subplots(2,3,figsize=(14,10))


# First plot
plot_idx = (0,0)
plot_title = "RRE history (MM weights)"
metric = "rre_history"


method = "GKS"
grid = [i for i in range(start_idx, len(v_gks_data[metric]))]
axs[plot_idx].plot( grid, v_gks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
#axs[plot_idx].scatter( grid, v_gks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method )

method = "S-GKS"
grid = [i for i in range(start_idx, len(mm_sgks_data[metric]))]
axs[plot_idx].plot( grid, mm_sgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
#axs[plot_idx].scatter( grid, mm_sgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)


method = "resS-GKS"
grid = [i for i in range(start_idx, len(mm_ressgks_data[metric]))]
axs[plot_idx].plot( grid, mm_ressgks_data[metric][start_idx:], color=method_colors["S-GKS"], ls="dotted", lw=line_width, label=method)


method = "recS-GKS"
grid = [i for i in range(start_idx, len(mm_recsgks_data[metric]))]
axs[plot_idx].plot( grid, mm_recsgks_data[metric][start_idx:], color=method_colors["S-GKS"], ls="dashed", lw=line_width, label=method)



method = "PS-GKS"
grid = [i for i in range(start_idx, len(mm_psgks_data[metric]))]
axs[plot_idx].plot( grid, mm_psgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
#axs[plot_idx].scatter( grid, mm_psgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)


method = "resPS-GKS"
grid = [i for i in range(start_idx, len(mm_respsgks_data[metric]))]
axs[plot_idx].plot( grid, mm_respsgks_data[metric][start_idx:], color=method_colors["PS-GKS"], ls="dotted", lw=line_width, label=method)
#axs[plot_idx].scatter( grid, mm_gkb_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)


method = "recPS-GKS"
grid = [i for i in range(start_idx, len(mm_recpsgks_data[metric]))]
axs[plot_idx].plot( grid, mm_recpsgks_data[metric][start_idx:], color=method_colors["PS-GKS"], ls="dashed", lw=line_width, label=method)
#axs[plot_idx].scatter( grid, mm_gkb_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)




method = "PS-GKB"
grid = [i for i in range(start_idx, len(mm_gkb_data[metric]))]
axs[plot_idx].plot( grid, mm_gkb_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
#axs[plot_idx].scatter( grid, mm_gkb_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)




method = "FGK"
grid = [i for i in range(start_idx, len(mm_fgk_data[metric]))]
axs[plot_idx].plot( grid, mm_fgk_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
#axs[plot_idx].scatter( grid, mm_fgk_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)


# method = "IRW-FLSQR"
# grid = [i for i in range(start_idx, len(mm_fgk2019IRW_data[metric]))]
# axs[plot_idx].plot( grid, mm_fgk2019IRW_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
# #axs[plot_idx].scatter( grid, mm_fgk2019IRW_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)


# method = "FLSQR-R"
# grid = [i for i in range(start_idx, len(mm_fgk2019R_data[metric]))]
# axs[plot_idx].plot( grid, mm_fgk2019R_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
# #axs[plot_idx].scatter( grid, mm_fgk2019W_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=2*marker_size, label=method)


# method = "FLSQR-W"
# grid = [i for i in range(start_idx, len(mm_fgk2019W_data[metric]))]
# axs[plot_idx].plot( grid, mm_fgk2019W_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
# #axs[plot_idx].scatter( grid, mm_fgk2019W_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=2*marker_size, label=method)



# method = "FLSQR-I"
# grid = [i for i in range(start_idx, len(mm_fgk2019I_data[metric]))]
# axs[plot_idx].plot( grid, mm_fgk2019I_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width)
# axs[plot_idx].scatter( grid, mm_fgk2019I_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=2*marker_size, label=method)


# method = "FLSQR-R"
# grid = [i for i in range(start_idx, len(mm_fgk2019R_data[metric]))]
# axs[plot_idx].plot( grid, mm_fgk2019R_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width)
# axs[plot_idx].scatter( grid, mm_fgk2019R_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=2*marker_size, label=method)




axs[plot_idx].set_xlabel("Iteration index")
axs[plot_idx].legend()
axs[plot_idx].set_title(plot_title)




# First plot
plot_idx = (1,0)
plot_title = "RRE history (IAS weights)"
metric = "rre_history"

method = "GKS"
grid = [i for i in range(start_idx, len(v_gks_data[metric]))]
axs[plot_idx].plot( grid, v_gks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
#axs[plot_idx].scatter( grid, v_gks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method )

method = "S-GKS"
grid = [i for i in range(start_idx, len(ias_sgks_data[metric]))]
axs[plot_idx].plot( grid, ias_sgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
#axs[plot_idx].scatter( grid, ias_sgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)


method = "resS-GKS"
grid = [i for i in range(start_idx, len(ias_ressgks_data[metric]))]
axs[plot_idx].plot( grid, ias_ressgks_data[metric][start_idx:], color=method_colors["S-GKS"], ls="dotted", lw=line_width, label=method)
#axs[plot_idx].scatter( grid, ias_sgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)

method = "recS-GKS"
grid = [i for i in range(start_idx, len(ias_recsgks_data[metric]))]
axs[plot_idx].plot( grid, ias_recsgks_data[metric][start_idx:], color=method_colors["S-GKS"], ls="dashed", lw=line_width, label=method)



method = "PS-GKS"
grid = [i for i in range(start_idx, len(ias_psgks_data[metric]))]
axs[plot_idx].plot( grid, ias_psgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
#axs[plot_idx].scatter( grid, ias_psgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)


method = "resPS-GKS"
grid = [i for i in range(start_idx, len(ias_respsgks_data[metric]))]
axs[plot_idx].plot( grid, ias_respsgks_data[metric][start_idx:], color=method_colors["PS-GKS"], ls="dotted", lw=line_width, label=method)
#axs[plot_idx].scatter( grid, ias_psgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)


method = "recPS-GKS"
grid = [i for i in range(start_idx, len(ias_recpsgks_data[metric]))]
axs[plot_idx].plot( grid, ias_recpsgks_data[metric][start_idx:], color=method_colors["PS-GKS"], ls="dashed", lw=line_width, label=method)
#axs[plot_idx].scatter( grid, ias_psgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)



method = "PS-GKB"
grid = [i for i in range(start_idx, len(ias_gkb_data[metric]))]
axs[plot_idx].plot( grid, ias_gkb_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
#axs[plot_idx].scatter( grid, ias_psgkb_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)


method = "FGK"
grid = [i for i in range(start_idx, len(ias_fgk_data[metric]))]
axs[plot_idx].plot( grid, ias_fgk_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
#axs[plot_idx].scatter( grid, ias_fgk_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)



# method = "IRW-FLSQR"
# grid = [i for i in range(start_idx, len(ias_fgk2019IRW_data[metric]))]
# axs[plot_idx].plot( grid, ias_fgk2019IRW_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
# #axs[plot_idx].scatter( grid, ias_fgk2019IRW_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=2*marker_size, label=method)


# method = "FLSQR-R"
# grid = [i for i in range(start_idx, len(ias_fgk2019R_data[metric]))]
# axs[plot_idx].plot( grid, ias_fgk2019R_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
# #axs[plot_idx].scatter( grid, ias_fgk2019R_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=2*marker_size, label=method)


# method = "FLSQR-W"
# grid = [i for i in range(start_idx, len(ias_fgk2019W_data[metric]))]
# axs[plot_idx].plot( grid, ias_fgk2019W_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
# #axs[plot_idx].scatter( grid, ias_fgk2019W_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=2*marker_size, label=method)


# method = "FLSQR-I"
# grid = [i for i in range(start_idx, len(ias_fgk2019I_data[metric]))]
# axs[plot_idx].plot( grid, ias_fgk2019I_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width)
# axs[plot_idx].scatter( grid, ias_fgk2019I_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=2*marker_size, label=method)


axs[plot_idx].set_xlabel("Iteration index")
axs[plot_idx].legend()
axs[plot_idx].set_title(plot_title)









# First plot
plot_idx = (0,1)
plot_title = "SSIM history (MM weights)"
metric = "ssim_history"

method = "GKS"
grid = [i for i in range(start_idx, len(v_gks_data[metric]))]
axs[plot_idx].plot( grid, v_gks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
#axs[plot_idx].scatter( grid, v_gks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method )

method = "S-GKS"
grid = [i for i in range(start_idx, len(mm_sgks_data[metric]))]
axs[plot_idx].plot( grid, mm_sgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
#axs[plot_idx].scatter( grid, mm_sgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)

method = "resS-GKS"
grid = [i for i in range(start_idx, len(mm_ressgks_data[metric]))]
axs[plot_idx].plot( grid, mm_ressgks_data[metric][start_idx:], color=method_colors["S-GKS"], ls="dotted", lw=line_width, label=method)
#axs[plot_idx].scatter( grid, mm_sgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)

method = "recS-GKS"
grid = [i for i in range(start_idx, len(mm_recsgks_data[metric]))]
axs[plot_idx].plot( grid, mm_recsgks_data[metric][start_idx:], color=method_colors["S-GKS"], ls="dashed", lw=line_width, label=method)
#axs[plot_idx].scatter( grid, mm_sgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)

method = "PS-GKS"
grid = [i for i in range(start_idx, len(mm_psgks_data[metric]))]
axs[plot_idx].plot( grid, mm_psgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
#axs[plot_idx].scatter( grid, mm_psgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)

method = "resPS-GKS"
grid = [i for i in range(start_idx, len(mm_respsgks_data[metric]))]
axs[plot_idx].plot( grid, mm_respsgks_data[metric][start_idx:], color=method_colors["PS-GKS"], ls="dotted", lw=line_width, label=method)
#axs[plot_idx].scatter( grid, mm_psgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)


method = "recPS-GKS"
grid = [i for i in range(start_idx, len(mm_recpsgks_data[metric]))]
axs[plot_idx].plot( grid, mm_recpsgks_data[metric][start_idx:], color=method_colors["PS-GKS"], ls="dashed", lw=line_width, label=method)
#axs[plot_idx].scatter( grid, mm_psgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)



method = "PS-GKB"
grid = [i for i in range(start_idx, len(mm_gkb_data[metric]))]
axs[plot_idx].plot( grid, mm_gkb_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
#axs[plot_idx].scatter( grid, mm_psgkb_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)


method = "FGK"
grid = [i for i in range(start_idx, len(mm_fgk_data[metric]))]
axs[plot_idx].plot( grid, mm_fgk_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
#axs[plot_idx].scatter( grid, mm_fgk_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)


# method = "IRW-FLSQR"
# grid = [i for i in range(start_idx, len(mm_fgk2019IRW_data[metric]))]
# axs[plot_idx].plot( grid, mm_fgk2019IRW_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
# #axs[plot_idx].scatter( grid, mm_fgk2019IRW_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)


# method = "FLSQR-R"
# grid = [i for i in range(start_idx, len(mm_fgk2019R_data[metric]))]
# axs[plot_idx].plot( grid, mm_fgk2019R_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
# #axs[plot_idx].scatter( grid, mm_fgk2019W_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=2*marker_size, label=method)


# method = "FLSQR-W"
# grid = [i for i in range(start_idx, len(mm_fgk2019W_data[metric]))]
# axs[plot_idx].plot( grid, mm_fgk2019W_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
# #axs[plot_idx].scatter( grid, mm_fgk2019W_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=2*marker_size, label=method)


axs[plot_idx].set_xlabel("Iteration index")
axs[plot_idx].legend()
axs[plot_idx].set_title(plot_title)






# First plot
plot_idx = (1,1)
plot_title = "SSIM history (IAS weights)"
metric = "ssim_history"

method = "GKS"
grid = [i for i in range(start_idx, len(v_gks_data[metric]))]
axs[plot_idx].plot( grid, v_gks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
#axs[plot_idx].scatter( grid, v_gks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method )

method = "S-GKS"
grid = [i for i in range(start_idx, len(ias_sgks_data[metric]))]
axs[plot_idx].plot( grid, ias_sgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
#axs[plot_idx].scatter( grid, ias_sgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)

method = "resS-GKS)"
grid = [i for i in range(start_idx, len(ias_ressgks_data[metric]))]
axs[plot_idx].plot( grid, ias_ressgks_data[metric][start_idx:], color=method_colors["S-GKS"], ls="dotted", lw=line_width, label=method)
#axs[plot_idx].scatter( grid, ias_sgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)

method = "recS-GKS"
grid = [i for i in range(start_idx, len(ias_recsgks_data[metric]))]
axs[plot_idx].plot( grid, ias_recsgks_data[metric][start_idx:], color=method_colors["S-GKS"], ls="dashed", lw=line_width, label=method)
#axs[plot_idx].scatter( grid, ias_sgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)


method = "PS-GKS"
grid = [i for i in range(start_idx, len(ias_psgks_data[metric]))]
axs[plot_idx].plot( grid, ias_psgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
#axs[plot_idx].scatter( grid, ias_psgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)

method = "resPS-GKS"
grid = [i for i in range(start_idx, len(ias_respsgks_data[metric]))]
axs[plot_idx].plot( grid, ias_respsgks_data[metric][start_idx:], color=method_colors["PS-GKS"], ls="dotted", lw=line_width, label=method)
#axs[plot_idx].scatter( grid, ias_psgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)


method = "recPS-GKS"
grid = [i for i in range(start_idx, len(ias_recpsgks_data[metric]))]
axs[plot_idx].plot( grid, ias_recpsgks_data[metric][start_idx:], color=method_colors["PS-GKS"], ls="dashed", lw=line_width, label=method)
#axs[plot_idx].scatter( grid, ias_psgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)



method = "PS-GKB"
grid = [i for i in range(start_idx, len(ias_gkb_data[metric]))]
axs[plot_idx].plot( grid, ias_gkb_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
#axs[plot_idx].scatter( grid, ias_psgkb_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)


method = "FGK"
grid = [i for i in range(start_idx, len(ias_fgk_data[metric]))]
axs[plot_idx].plot( grid, ias_fgk_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
#axs[plot_idx].scatter( grid, ias_fgk_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)


# method = "IRW-FLSQR"
# grid = [i for i in range(start_idx, len(ias_fgk2019IRW_data[metric]))]
# axs[plot_idx].plot( grid, ias_fgk2019IRW_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
# #axs[plot_idx].scatter( grid, ias_fgk2019IRW_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=2*marker_size, label=method)


# method = "FLSQR-R"
# grid = [i for i in range(start_idx, len(ias_fgk2019R_data[metric]))]
# axs[plot_idx].plot( grid, ias_fgk2019R_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
# #axs[plot_idx].scatter( grid, ias_fgk2019W_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=2*marker_size, label=method)


# method = "FLSQR-W"
# grid = [i for i in range(start_idx, len(ias_fgk2019W_data[metric]))]
# axs[plot_idx].plot( grid, ias_fgk2019W_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
# #axs[plot_idx].scatter( grid, ias_fgk2019W_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=2*marker_size, label=method)


axs[plot_idx].set_xlabel("Iteration index")
axs[plot_idx].legend(loc="lower right")
axs[plot_idx].set_title(plot_title)












# First plot
plot_idx = (0,2)
plot_title = "Gini index (MM weights)"
metric = "gini_indices"

method = "GKS"
grid = [i for i in range(start_idx, len(v_gks_data[metric]))]
axs[plot_idx].plot( grid, v_gks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
#axs[plot_idx].scatter( grid, v_gks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method )

method = "S-GKS"
grid = [i for i in range(start_idx, len(mm_sgks_data[metric]))]
axs[plot_idx].plot( grid, mm_sgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
#axs[plot_idx].scatter( grid, mm_sgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)

method = "resS-GKS"
grid = [i for i in range(start_idx, len(mm_ressgks_data[metric]))]
axs[plot_idx].plot( grid, mm_ressgks_data[metric][start_idx:], color=method_colors["S-GKS"], ls="dotted", lw=line_width, label=method)
#axs[plot_idx].scatter( grid, mm_sgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)

method = "recS-GKS"
grid = [i for i in range(start_idx, len(mm_recsgks_data[metric]))]
axs[plot_idx].plot( grid, mm_recsgks_data[metric][start_idx:], color=method_colors["S-GKS"], ls="dashed", lw=line_width, label=method)
#axs[plot_idx].scatter( grid, mm_sgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)


method = "PS-GKS"
grid = [i for i in range(start_idx, len(mm_psgks_data[metric]))]
axs[plot_idx].plot( grid, mm_psgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
#axs[plot_idx].scatter( grid, mm_psgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)


method = "resPS-GKS"
grid = [i for i in range(start_idx, len(mm_respsgks_data[metric]))]
axs[plot_idx].plot( grid, mm_respsgks_data[metric][start_idx:], color=method_colors["PS-GKS"], ls="dotted", lw=line_width, label=method)
#axs[plot_idx].scatter( grid, mm_psgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)

method = "recPS-GKS"
grid = [i for i in range(start_idx, len(mm_recpsgks_data[metric]))]
axs[plot_idx].plot( grid, mm_recpsgks_data[metric][start_idx:], color=method_colors["PS-GKS"], ls="dashed", lw=line_width, label=method)
#axs[plot_idx].scatter( grid, mm_psgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)



method = "PS-GKB"
grid = [i for i in range(start_idx, len(mm_gkb_data[metric]))]
axs[plot_idx].plot( grid, mm_gkb_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
#axs[plot_idx].scatter( grid, mm_psgkb_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)


method = "FGK"
grid = [i for i in range(start_idx, len(mm_fgk_data[metric]))]
axs[plot_idx].plot( grid, mm_fgk_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
#axs[plot_idx].scatter( grid, mm_fgk_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)



# method = "IRW-FLSQR"
# grid = [i for i in range(start_idx, len(mm_fgk2019IRW_data[metric]))]
# axs[plot_idx].plot( grid, mm_fgk2019IRW_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
# #axs[plot_idx].scatter( grid, mm_fgk2019IRW_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)


# method = "FLSQR-R"
# grid = [i for i in range(start_idx, len(mm_fgk2019R_data[metric]))]
# axs[plot_idx].plot( grid, mm_fgk2019R_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
# #axs[plot_idx].scatter( grid, mm_fgk2019W_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=2*marker_size, label=method)


# method = "FLSQR-W"
# grid = [i for i in range(start_idx, len(mm_fgk2019W_data[metric]))]
# axs[plot_idx].plot( grid, mm_fgk2019W_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
# #axs[plot_idx].scatter( grid, mm_fgk2019W_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=2*marker_size, label=method)




axs[plot_idx].set_xlabel("Iteration index")
axs[plot_idx].legend()
axs[plot_idx].set_title(plot_title)




# First plot
plot_idx = (1,2)
plot_title = "Gini index (IAS weights)"
metric = "gini_indices"

method = "GKS"
grid = [i for i in range(start_idx, len(v_gks_data[metric]))]
axs[plot_idx].plot( grid, v_gks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
#axs[plot_idx].scatter( grid, v_gks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method )

method = "S-GKS"
grid = [i for i in range(start_idx, len(ias_sgks_data[metric]))]
axs[plot_idx].plot( grid, ias_sgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
#axs[plot_idx].scatter( grid, ias_sgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)


method = "resS-GKS"
grid = [i for i in range(start_idx, len(ias_ressgks_data[metric]))]
axs[plot_idx].plot( grid, ias_ressgks_data[metric][start_idx:], color=method_colors["S-GKS"], ls="dotted", lw=line_width, label=method)
#axs[plot_idx].scatter( grid, ias_sgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)

method = "recS-GKS"
grid = [i for i in range(start_idx, len(ias_recsgks_data[metric]))]
axs[plot_idx].plot( grid, ias_recsgks_data[metric][start_idx:], color=method_colors["S-GKS"], ls="dashed", lw=line_width, label=method)
#axs[plot_idx].scatter( grid, ias_sgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)


method = "PS-GKS"
grid = [i for i in range(start_idx, len(ias_psgks_data[metric]))]
axs[plot_idx].plot( grid, ias_psgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
#axs[plot_idx].scatter( grid, ias_psgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)

method = "resPS-GKS"
grid = [i for i in range(start_idx, len(ias_respsgks_data[metric]))]
axs[plot_idx].plot( grid, ias_respsgks_data[metric][start_idx:], color=method_colors["PS-GKS"], ls="dotted", lw=line_width, label=method)
#axs[plot_idx].scatter( grid, ias_psgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)

method = "recPS-GKS"
grid = [i for i in range(start_idx, len(ias_recpsgks_data[metric]))]
axs[plot_idx].plot( grid, ias_recpsgks_data[metric][start_idx:], color=method_colors["PS-GKS"], ls="dashed", lw=line_width, label=method)
#axs[plot_idx].scatter( grid, ias_psgks_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)



method = "PS-GKB"
grid = [i for i in range(start_idx, len(ias_gkb_data[metric]))]
axs[plot_idx].plot( grid, ias_gkb_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
#axs[plot_idx].scatter( grid, ias_psgkb_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)


method = "FGK"
grid = [i for i in range(start_idx, len(ias_fgk_data[metric]))]
axs[plot_idx].plot( grid, ias_fgk_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
#axs[plot_idx].scatter( grid, ias_fgk_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=marker_size, label=method)



# method = "IRW-FLSQR"
# grid = [i for i in range(start_idx, len(ias_fgk2019IRW_data[metric]))]
# axs[plot_idx].plot( grid, ias_fgk2019IRW_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
# #axs[plot_idx].scatter( grid, ias_fgk2019IRW_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=2*marker_size, label=method)

# method = "FLSQR-R"
# grid = [i for i in range(start_idx, len(ias_fgk2019R_data[metric]))]
# axs[plot_idx].plot( grid, ias_fgk2019R_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
# #axs[plot_idx].scatter( grid, ias_fgk2019W_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=2*marker_size, label=method)

# method = "FLSQR-W"
# grid = [i for i in range(start_idx, len(ias_fgk2019W_data[metric]))]
# axs[plot_idx].plot( grid, ias_fgk2019W_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], lw=line_width, label=method)
# #axs[plot_idx].scatter( grid, ias_fgk2019W_data[metric][start_idx:], color=method_colors[method], ls=method_lines[method], marker=method_markers[method], s=2*marker_size, label=method)



axs[plot_idx].set_xlabel("Iteration index")
axs[plot_idx].legend()
axs[plot_idx].set_title(plot_title)



fig.tight_layout()
fig.savefig(RESULTS / "figure_09.png", dpi=250, bbox_inches=None)




save_curve_data(fig, RESULTS / "figure_09.csv")
plt.show()
plt.close(fig)


The CSV files contain the values from this run. The `.npz` files contain numeric arrays only and can be opened with `np.load(..., allow_pickle=False)`. Individual solver results are saved as each method finishes.
